<a href="https://colab.research.google.com/github/DavidTan-cloud/wqf7023-industrial-anomaly-detection-radar/blob/main/notebooks/02_smap_lstm_ae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/ari-dasci/RADAR.git
%cd RADAR
!pip install -r requirements.txt

Cloning into 'RADAR'...
remote: Enumerating objects: 993, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 993 (delta 33), reused 4 (delta 2), pack-reused 921 (from 1)
Receiving objects: 100% (993/993), 28.90 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (650/650), done.
/content/RADAR
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.6/169.6 kB 14.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

In [4]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

sys.path.append('/content/RADAR')

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.11.0+cpu
CUDA Available: False


In [6]:
import os

print(os.listdir('/content/RADAR/RADAR'))

['static_data', '__init__.py', 'base_preprocessing_module.py', 'pos_process_module.py', 'visualization_module.py', 'federated_data', 'base_utils_module.py', 'metrics_module.py', 'time_series', 'base_algorithm_module.py']


In [7]:
!find /content/RADAR/RADAR -type f | head -100

/content/RADAR/RADAR/static_data/__init__.py
/content/RADAR/RADAR/static_data/preprocessing/__init__.py
/content/RADAR/RADAR/static_data/preprocessing/preprocessing_static.py
/content/RADAR/RADAR/static_data/static_datasets_uci.py
/content/RADAR/RADAR/static_data/anomaly_dataset_utils.py
/content/RADAR/RADAR/static_data/algorithms/__init__.py
/content/RADAR/RADAR/static_data/algorithms/sklearn.py
/content/RADAR/RADAR/static_data/algorithms/pyod.py
/content/RADAR/RADAR/__init__.py
/content/RADAR/RADAR/base_preprocessing_module.py
/content/RADAR/RADAR/pos_process_module.py
/content/RADAR/RADAR/visualization_module.py
/content/RADAR/RADAR/federated_data/__init__.py
/content/RADAR/RADAR/federated_data/algorithms/__init__.py
/content/RADAR/RADAR/federated_data/algorithms/flexanomalies.py
/content/RADAR/RADAR/base_utils_module.py
/content/RADAR/RADAR/metrics_module.py
/content/RADAR/RADAR/time_series/__init__.py
/content/RADAR/RADAR/time_series/preprocessing/__init__.py
/content/RADAR/RADAR/

In [8]:
time_series_component_path = '/content/RADAR/RADAR/time_series'
print(time_series_component_path)

/content/RADAR/RADAR/time_series


In [11]:
# Install PyOD and scikit-learn
!pip install pyod scikit-learn

# Verify installations
import pyod
import sklearn

print(f"PyOD version: {pyod.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")

# The other components (TSFE-DL, Informer, Autoformer, Vanilla Transformer, flex-anomalies)
# are either already installed via requirements.txt or are part of the RADAR project's code structure.
# No further pip installation is needed for them.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 11.4 MB/s eta 0:00:00
PyOD version: 3.6.1
Scikit-learn version: 1.6.1


In [8]:
import os
import re

# Paths to the problematic files
pyod_file_path = '/content/RADAR/RADAR/static_data/algorithms/pyod.py'
sklearn_file_path = '/content/RADAR/RADAR/static_data/algorithms/sklearn.py'
anomaly_dataset_utils_file_path = '/content/RADAR/RADAR/static_data/anomaly_dataset_utils.py'
preprocessing_static_file_path = '/content/RADAR/RADAR/static_data/preprocessing/preprocessing_static.py'
tsfedl_file_path = '/content/RADAR/RADAR/time_series/algorithms/tsfedl.py'

# --- Fix for pyod.py ---
with open(pyod_file_path, 'r') as f:
    content_pyod = f.read()

old_import_base = "from RADAR.base_algorithm_module import BaseAnomalyDetection"
new_import_base = "from RADAR.RADAR.base_algorithm_module import BaseAnomalyDetection"

if old_import_base in content_pyod:
    content_pyod = content_pyod.replace(old_import_base, new_import_base)
    print(f"Fixed import for BaseAnomalyDetection in {pyod_file_path}")
else:
    print(f"Import statement '{old_import_base}' not found in {pyod_file_path}. No changes made for BaseAnomalyDetection.")

old_import_metrics = "from RADAR.metrics_module import print_metrics"
new_import_metrics = "from RADAR.RADAR.metrics_module import print_metrics"

if old_import_metrics in content_pyod:
    content_pyod = content_pyod.replace(old_import_metrics, new_import_metrics)
    print(f"Fixed import for metrics_module in {pyod_file_path}")
else:
    print(f"Import statement '{old_import_metrics}' not found in {pyod_file_path}. No changes made for metrics_module.")

with open(pyod_file_path, 'w') as f:
    f.write(content_pyod)

# --- Fix for sklearn.py ---
with open(sklearn_file_path, 'r') as f:
    content_sklearn = f.read()

if old_import_base in content_sklearn:
    content_sklearn = content_sklearn.replace(old_import_base, new_import_base)
    print(f"Fixed import for BaseAnomalyDetection in {sklearn_file_path}")
else:
    print(f"Import statement '{old_import_base}' not found in {sklearn_file_path}. No changes made for BaseAnomalyDetection.")

if old_import_metrics in content_sklearn:
    content_sklearn = content_sklearn.replace(old_import_metrics, new_import_metrics)
    print(f"Fixed import for metrics_module in {sklearn_file_path}")
else:
    print(f"Import statement '{old_import_metrics}' not found in {sklearn_file_path}. No changes made for metrics_module.")

with open(sklearn_file_path, 'w') as f:
    f.write(content_sklearn)

# --- Fix for anomaly_dataset_utils.py ---
with open(anomaly_dataset_utils_file_path, 'r') as f:
    content_anomaly_utils = f.read()

# Specific replacements for anomaly_dataset_utils.py
old_import_prep = "from RADAR.static_data.preprocessing.preprocessing_static"
new_import_prep = "from RADAR.RADAR.static_data.preprocessing.preprocessing_static"

if old_import_prep in content_anomaly_utils:
    content_anomaly_utils = content_anomaly_utils.replace(old_import_prep, new_import_prep)
    print(f"Fixed import for preprocessing_static in {anomaly_dataset_utils_file_path}")
else:
    print(f"Import statement '{old_import_prep}' not found in {anomaly_dataset_utils_file_path}. No changes made.")

old_import_datasets = "from RADAR.static_data.static_datasets_uci"
new_import_datasets = "from RADAR.RADAR.static_data.static_datasets_uci"

if old_import_datasets in content_anomaly_utils:
    content_anomaly_utils = content_anomaly_utils.replace(old_import_datasets, new_import_datasets)
    print(f"Fixed import for static_datasets_uci in {anomaly_dataset_utils_file_path}")
else:
    print(f"Import statement '{old_import_datasets}' not found in {anomaly_dataset_utils_file_path}. No changes made.")

with open(anomaly_dataset_utils_file_path, 'w') as f:
    f.write(content_anomaly_utils)

# --- Fix for preprocessing_static.py ---
with open(preprocessing_static_file_path, 'r') as f:
    content_preprocessing_static = f.read()

old_import_base_preproc = "from RADAR.base_preprocessing_module import BasePreprocessing"
new_import_base_preproc = "from RADAR.RADAR.base_preprocessing_module import BasePreprocessing"

if old_import_base_preproc in content_preprocessing_static:
    content_preprocessing_static = content_preprocessing_static.replace(old_import_base_preproc, new_import_base_preproc)
    print(f"Fixed import for BasePreprocessing in {preprocessing_static_file_path}")
else:
    print(f"Import statement '{old_import_base_preproc}' not found in {preprocessing_static_file_path}. No changes made.")

with open(preprocessing_static_file_path, 'w') as f:
    f.write(content_preprocessing_static)

# --- Fix for tsfedl.py ---
with open(tsfedl_file_path, 'r') as f:
    content_tsfedl = f.read()

# Fix for BaseAnomalyDetection in tsfedl.py
if old_import_base in content_tsfedl:
    content_tsfedl = content_tsfedl.replace(old_import_base, new_import_base)
    print(f"Fixed import for BaseAnomalyDetection in {tsfedl_file_path}")
else:
    print(f"Import statement '{old_import_base}' not found in {tsfedl_file_path}. No changes made for BaseAnomalyDetection.")

# Fix for metrics_module in tsfedl.py
if old_import_metrics in content_tsfedl:
    content_tsfedl = content_tsfedl.replace(old_import_metrics, new_import_metrics)
    print(f"Fixed import for metrics_module in {tsfedl_file_path}")
else:
    print(f"Import statement '{old_import_metrics}' not found in {tsfedl_file_path}. No changes made for metrics_module.")

with open(tsfedl_file_path, 'w') as f:
    f.write(content_tsfedl)


# Continue with the original cell's code
!pip install combo ucimlrepo pytorch-lightning TSFEDL
from RADAR.RADAR.time_series import datasets_uci

# Load the SMAP dataset
smap_data = datasets_uci.load_dataset(dataset_name='SMAP')

print(f"SMAP dataset loaded successfully. Data structure: {type(smap_data)}")
# You might want to inspect the data further, e.g., smap_data.keys(), smap_data['data'].shape

Import statement 'from RADAR.base_algorithm_module import BaseAnomalyDetection' not found in /content/RADAR/RADAR/static_data/algorithms/pyod.py. No changes made for BaseAnomalyDetection.
Import statement 'from RADAR.metrics_module import print_metrics' not found in /content/RADAR/RADAR/static_data/algorithms/pyod.py. No changes made for metrics_module.
Import statement 'from RADAR.base_algorithm_module import BaseAnomalyDetection' not found in /content/RADAR/RADAR/static_data/algorithms/sklearn.py. No changes made for BaseAnomalyDetection.
Import statement 'from RADAR.metrics_module import print_metrics' not found in /content/RADAR/RADAR/static_data/algorithms/sklearn.py. No changes made for metrics_module.
Import statement 'from RADAR.static_data.preprocessing.preprocessing_static' not found in /content/RADAR/RADAR/static_data/anomaly_dataset_utils.py. No changes made.
Import statement 'from RADAR.static_data.static_datasets_uci' not found in /content/RADAR/RADAR/static_data/anomaly_

ModuleNotFoundError: No module named 'RADAR.base_algorithm_module'